In [ ]:
"""
E-commerce Web Scraping Script

This script is used to scrape the Amazon
Best Sellers page for Kitchen category and
store the results in a CSV file. The script
also creates a log file with the details of
the scraping process at each step of the code.

Author: Vinit Shah
Date: 26/02/2025
"""
import time
import csv
import logging
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from bs4 import BeautifulSoup

# Path to ChromeDriver
CHROMEDRIVER_PATH = "C:/Users/Rithin/OneDrive/Desktop/Python Programs/chromedriver-win64/chromedriver-win64/chromedriver.exe"

# Amazon Best Sellers URL
CATEGORY_URL = "https://www.amazon.in/gp/bestsellers/kitchen/ref=zg_bs_nav_kitchen_0"


In [6]:
# Configure logging
logging.basicConfig(
    filename="C:/Users/Rithin/OneDrive/Desktop/log.txt", 
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

In [7]:

def scrape_best_sellers(driver):
    logging.info("Starting Amazon Best Sellers Scraper")
    
    try:
        driver.get(CATEGORY_URL)
        time.sleep(3) 
        logging.info("Successfully loaded Amazon Best Sellers page.")

        soup = BeautifulSoup(driver.page_source, "html.parser")
        products = soup.select("div.p13n-gridRow div.p13n-sc-uncoverable-faceout")

        all_data = []
        for index, product in enumerate(products[:20], start=1):  # Limit to first 20 products
            try:
                name = product.select_one("div._cDEzb_p13n-sc-css-line-clamp-3_g3dy1").text.strip()
                price_tag = product.select_one("span._cDEzb_p13n-sc-price_3mJ9Z")
                price = price_tag.text.strip() if price_tag else "N/A"
                rating_tag = product.select_one("span.a-icon-alt")
                rating = rating_tag.text.split()[0] if rating_tag else "N/A"
                num_reviews_tag = product.select_one("span.a-size-small")
                num_reviews = num_reviews_tag.text.strip() if num_reviews_tag else "N/A"

                product_data = {
                    "Name": name,
                    "Price": price,
                    "Rating": rating,
                    "Reviews": num_reviews
                }
                all_data.append(product_data)

                logging.info(f"Scraped Product {index}: {name} | Price: {price} | Rating: {rating} | Reviews: {num_reviews}")

            except Exception as e:
                logging.error(f"Error scraping product {index}: {e}")

        # Save to CSV
        csv_path = "C:/Users/Rithin/OneDrive/Desktop/amazon_best_sellers.csv"
        try:
            with open(csv_path, "w", newline="", encoding="utf-8") as file:
                writer = csv.DictWriter(file, fieldnames=all_data[0].keys())
                writer.writeheader()
                writer.writerows(all_data)

            logging.info(f"Scraped data successfully saved to {csv_path}")

        except Exception as e:
            logging.error(f"Error saving CSV file: {e}")

    except Exception as e:
        logging.critical(f"Failed to scrape Amazon Best Sellers page: {e}")

In [8]:
def main():
    logging.info("Initializing WebDriver")
    service = Service(CHROMEDRIVER_PATH)
    options = webdriver.ChromeOptions()
    driver = webdriver.Chrome(service=service, options=options)

    try:
        scrape_best_sellers(driver)
    finally:
        driver.quit()
        logging.info("WebDriver closed successfully.")

if __name__ == "__main__":
    main()